# Tag 09 — Experten
## OOF Predictions, Blending & Multi-Level Stacking — Steel Industry Energy

Fixed version: exact `Steel_industry_data.csv`, column renaming, missing-value imputation, lighter execution.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

RANDOM_STATE = 42
OUTPUT_SUBDIR = "03_experten"
MAX_TRAIN_ROWS = 8000

In [ ]:
START = Path.cwd().resolve()
PROJECT_DIR = None
for p in [START, *START.parents]:
    if (p / "data" / "raw").exists() and (p / "outputs").exists():
        PROJECT_DIR = p
        break

if PROJECT_DIR is None:
    PROJECT_DIR = Path("C:/Users/esmae/Documents/Educx Kurs machine lerning/aufgabe/Tag_09_Ensemble_Methods_Project")

DATA_DIR = PROJECT_DIR / "data" / "raw"
OUTPUT_DIR = PROJECT_DIR / "outputs" / OUTPUT_SUBDIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
DATA_PATH = DATA_DIR / "Steel_industry_data.csv"

if not DATA_PATH.exists():
    possible_files = [
        "Steel_industry_data.csv",
        "steel_industry_data.csv",
        "Steel Industry Energy.csv",
        "steel_energy.csv",
        "Steel_industry_data.xlsx",
        "steel_industry_data.xlsx"
    ]
    DATA_PATH = None
    for file_name in possible_files:
        candidate = DATA_DIR / file_name
        if candidate.exists():
            DATA_PATH = candidate
            break

if DATA_PATH is None or not DATA_PATH.exists():
    raise FileNotFoundError(
        "Steel Industry Energy data not found. Put Steel_industry_data.csv in: " + str(DATA_DIR)
    )

print("Using file:", DATA_PATH)

if DATA_PATH.suffix.lower() in [".xlsx", ".xls"]:
    df = pd.read_excel(DATA_PATH)
else:
    df = pd.read_csv(DATA_PATH, encoding="latin1")

df.columns = df.columns.str.strip()

df = df.rename(columns={
    "date": "Datum",
    "WeekStatus": "wochenstatus",
    "Day_of_week": "wochentag",
    "Load_Type": "lasttyp",
    "Load Type": "lasttyp",
    "LoadType": "lasttyp"
})

print("Data loaded successfully")
print("Shape:", df.shape)
print(df.head())
print(df.columns.tolist())

In [ ]:
target_col = "lasttyp"
if target_col not in df.columns:
    raise ValueError(
        "Target column lasttyp not found. Available columns: " + str(df.columns.tolist())
    )

work = df.dropna(subset=[target_col]).copy()

if "Datum" in work.columns:
    work["Datum"] = pd.to_datetime(work["Datum"], errors="coerce", dayfirst=True)
    work["Datum_month"] = work["Datum"].dt.month
    work["Datum_dayofweek"] = work["Datum"].dt.dayofweek
    work["Datum_hour"] = work["Datum"].dt.hour
    work = work.drop(columns=["Datum"])

X = work.drop(columns=[target_col])
y = work[target_col].astype(str).str.strip()

for col in X.columns:
    if X[col].dtype == "object":
        X[col] = X[col].astype(str).str.strip()

print("X shape:", X.shape)
print("Class distribution:")
print(y.value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

if len(X_train) > MAX_TRAIN_ROWS:
    X_train_work, _, y_train_work, _ = train_test_split(
        X_train,
        y_train,
        train_size=MAX_TRAIN_ROWS,
        random_state=RANDOM_STATE,
        stratify=y_train
    )
else:
    X_train_work = X_train.copy()
    y_train_work = y_train.copy()

X_train_work = X_train_work.reset_index(drop=True)
y_train_work = y_train_work.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

classes = np.array(sorted(y_train_work.unique()))

numeric_features = X_train_work.select_dtypes(include=["int64", "float64", "int32", "float32", "bool"]).columns.tolist()
categorical_features = X_train_work.select_dtypes(include=["object", "category"]).columns.tolist()

print("Train work:", X_train_work.shape)
print("Test:", X_test.shape)
print("Classes:", classes)
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

In [ ]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

base_estimators = {
    "rf": Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(n_estimators=80, random_state=RANDOM_STATE, n_jobs=1))
    ]),
    "lr": Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ]),
    "knn": Pipeline([
        ("preprocessor", preprocessor),
        ("model", KNeighborsClassifier(n_neighbors=7))
    ]),
    "dt": Pipeline([
        ("preprocessor", preprocessor),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE))
    ])
}

print("Base estimators are ready")

In [ ]:
def aligned_proba(model, X_part, classes):
    proba = model.predict_proba(X_part)
    model_classes = model.classes_
    out = np.zeros((len(X_part), len(classes)))
    for i, c in enumerate(model_classes):
        j = np.where(classes == c)[0][0]
        out[:, j] = proba[:, i]
    return out


def build_oof_features(base_estimators, X_train, y_train, X_test, classes, n_splits=3):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof_parts = []
    test_parts = []
    details = []

    for name, estimator in base_estimators.items():
        print("Building OOF features for:", name)
        oof = np.zeros((len(X_train), len(classes)))
        fold_test_predictions = []

        for fold, (tr_idx, val_idx) in enumerate(cv.split(X_train, y_train), start=1):
            model = clone(estimator)
            model.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
            oof[val_idx] = aligned_proba(model, X_train.iloc[val_idx], classes)
            fold_test_predictions.append(aligned_proba(model, X_test, classes))
            details.append({"base_model": name, "fold": fold})

        test_mean = np.mean(fold_test_predictions, axis=0)
        oof_parts.append(oof)
        test_parts.append(test_mean)

    meta_train = np.hstack(oof_parts)
    meta_test = np.hstack(test_parts)
    return meta_train, meta_test, pd.DataFrame(details)

meta_train, meta_test, oof_details = build_oof_features(
    base_estimators,
    X_train_work,
    y_train_work,
    X_test,
    classes,
    n_splits=3
)

print("meta_train:", meta_train.shape)
print("meta_test:", meta_test.shape)
oof_details.head()

In [ ]:
meta_model = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
meta_model.fit(meta_train, y_train_work)

oof_stack_pred = meta_model.predict(meta_test)
oof_stack_acc = accuracy_score(y_test, oof_stack_pred)

print("Manual OOF Stacking Test Accuracy:", round(oof_stack_acc, 4))
print(classification_report(y_test, oof_stack_pred))

cm = confusion_matrix(y_test, oof_stack_pred, labels=classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
disp.plot(cmap="Blues", xticks_rotation=45)
plt.title("Manual OOF Stacking - Confusion Matrix")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "expert_oof_stacking_confusion_matrix.png", dpi=200)
plt.show()

In [ ]:
X_subtrain, X_blend, y_subtrain, y_blend = train_test_split(
    X_train_work,
    y_train_work,
    test_size=0.3,
    random_state=RANDOM_STATE,
    stratify=y_train_work
)

blend_train_parts = []
blend_test_parts = []

for name, estimator in base_estimators.items():
    print("Training blender base model:", name)
    model = clone(estimator)
    model.fit(X_subtrain, y_subtrain)
    blend_train_parts.append(aligned_proba(model, X_blend, classes))
    blend_test_parts.append(aligned_proba(model, X_test, classes))

blend_meta_train = np.hstack(blend_train_parts)
blend_meta_test = np.hstack(blend_test_parts)

blend_meta_model = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
blend_meta_model.fit(blend_meta_train, y_blend)

blend_pred = blend_meta_model.predict(blend_meta_test)
blend_acc = accuracy_score(y_test, blend_pred)

print("Blending Test Accuracy:", round(blend_acc, 4))
print(classification_report(y_test, blend_pred))

In [ ]:
level1_models = {
    "level1_lr": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "level1_rf": RandomForestClassifier(n_estimators=80, random_state=RANDOM_STATE, n_jobs=1)
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
level1_oof_parts = []
level1_test_parts = []

meta_train_df = pd.DataFrame(meta_train)
meta_test_df = pd.DataFrame(meta_test)

for name, estimator in level1_models.items():
    print("Training level-1 model:", name)
    oof = np.zeros((len(meta_train_df), len(classes)))
    test_fold_preds = []

    for tr_idx, val_idx in cv.split(meta_train_df, y_train_work):
        model = clone(estimator)
        model.fit(meta_train_df.iloc[tr_idx], y_train_work.iloc[tr_idx])
        oof[val_idx] = aligned_proba(model, meta_train_df.iloc[val_idx], classes)
        test_fold_preds.append(aligned_proba(model, meta_test_df, classes))

    level1_oof_parts.append(oof)
    level1_test_parts.append(np.mean(test_fold_preds, axis=0))

level2_train = np.hstack(level1_oof_parts)
level2_test = np.hstack(level1_test_parts)

level2_model = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
level2_model.fit(level2_train, y_train_work)

three_layer_pred = level2_model.predict(level2_test)
three_layer_acc = accuracy_score(y_test, three_layer_pred)

print("Three-layer Stacking Test Accuracy:", round(three_layer_acc, 4))
print(classification_report(y_test, three_layer_pred))

In [ ]:
comparison = pd.DataFrame([
    {"method": "Manual OOF Stacking", "test_accuracy": oof_stack_acc},
    {"method": "Blending", "test_accuracy": blend_acc},
    {"method": "Three-layer Stacking", "test_accuracy": three_layer_acc}
]).sort_values("test_accuracy", ascending=False)

comparison.to_csv(OUTPUT_DIR / "expert_oof_blending_three_layer_comparison.csv", index=False)

plt.figure(figsize=(9, 5))
plt.bar(comparison["method"], comparison["test_accuracy"])
plt.xticks(rotation=25, ha="right")
plt.ylabel("Test Accuracy")
plt.title("OOF Stacking vs Blending vs Three-layer Stacking")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "expert_stacking_methods_comparison.png", dpi=200)
plt.show()

comparison